In [ ]:

"""
CN7 02 - 1425 -> 1211 필터 규칙 탐색 + 원본행 복원
=====================================================

목표
----
1) moldset_labeled.csv 에서 CN7 1425행 추출
2) moldset_labeled_cn7.csv 의 1211행이 어떤 원본 subset인지 탐색
3) StandardScaler(ddof=0)로 재현 가능한지 검증
4) 원본 metadata(TimeStamp, PART_NAME, PART_FACT_SERIAL 등) 복원
5) RH/LH Side 추출
6) PassOrFail 0/1이 원본과 동일/반전인지 검증

중요
----
- 기존 shot/pair 가설을 사용하지 않는다.
- 1=불량이라고 가정하지 않는다.
- 먼저 "어떤 1211개 행인지"를 데이터로 복원한다.
"""

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 경로
# =============================================================================

BASE_DIR = Path(
    r"../../../data/raw"
)

OUT_DIR = Path("cn7_02_restore")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NAMES = [
    "moldset_labeled.csv",
    "moldset labeled.csv",
]

TARGET_NAMES = [
    "moldset_labeled_cn7.csv",
    "moldset labeled cn7.csv",
]

TARGET_CANDIDATES = ["PassOrFail", "passorfail", "PASSORFAIL"]

INDEX_LIKE = {"unnamed: 0", "index", "_id", "id"}

# Hungarian assignment에서 label 불일치를 얼마나 강하게 막을지
LABEL_MISMATCH_PENALTY = 1_000.0

# 재현이 사실상 완벽하다고 볼 거리 기준
EXACT_DISTANCE = 1e-4

# iterative assignment 최대 반복
MAX_ITER = 20


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 105)
    print(title)
    print("=" * 105)


def norm(x):
    return str(x).strip().lower()


def find_file(base_dir, names):
    names = {x.lower() for x in names}
    for p in base_dir.rglob("*"):
        if p.is_file() and p.name.lower() in names:
            return p
    return None


def read_csv_flexible(path):
    last = None
    for enc in [None, "utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            if enc is None:
                return pd.read_csv(path)
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n{last}")


def find_col(df, candidates):
    m = {norm(c): c for c in df.columns}
    for x in candidates:
        if norm(x) in m:
            return m[norm(x)]
    return None


def target_col(df):
    return find_col(df, TARGET_CANDIDATES)


def get_cn7_mask(df):
    # 가장 직접적인 PART_NAME 우선
    part_name = find_col(df, ["PART_NAME"])
    if part_name is not None:
        return df[part_name].astype(str).str.contains(
            "CN7", case=False, na=False, regex=False
        )

    # fallback: 모든 컬럼
    mask = pd.Series(False, index=df.index)
    for c in df.columns:
        try:
            mask |= df[c].astype(str).str.contains(
                "CN7", case=False, na=False, regex=False
            )
        except Exception:
            pass
    return mask


def extract_side(series):
    s = series.astype(str).str.upper()
    out = pd.Series("UNKNOWN", index=series.index, dtype="object")
    out[s.str.contains(r"\bRH\b", regex=True, na=False)] = "RH"
    out[s.str.contains(r"\bLH\b", regex=True, na=False)] = "LH"
    return out


def common_numeric_features(source, target):
    t_src = target_col(source)
    t_tgt = target_col(target)

    target_map = {norm(c): c for c in target.columns}

    pairs = []
    for cs in source.columns:
        key = norm(cs)

        if key in INDEX_LIKE:
            continue
        if cs == t_src:
            continue
        if key not in target_map:
            continue

        ct = target_map[key]

        if ct == t_tgt:
            continue

        if (
            pd.api.types.is_numeric_dtype(source[cs])
            and pd.api.types.is_numeric_dtype(target[ct])
        ):
            pairs.append((cs, ct))

    return pairs


def scale_subset(subset, feature_cols):
    """
    sklearn StandardScaler와 같은 ddof=0.
    상수 컬럼은 0으로 출력.
    """
    X = subset[feature_cols].astype(float)
    mu = X.mean()
    sd = X.std(ddof=0)

    Z = pd.DataFrame(index=X.index, columns=feature_cols, dtype=float)

    for c in feature_cols:
        if pd.isna(sd[c]) or abs(sd[c]) < 1e-12:
            Z[c] = 0.0
        else:
            Z[c] = (X[c] - mu[c]) / sd[c]

    return Z, mu, sd


def direct_error(Z, T):
    A = Z.to_numpy(float)
    B = T.to_numpy(float)

    if A.shape != B.shape:
        return None

    d = np.sqrt(np.nansum((A - B) ** 2, axis=1))

    return {
        "mean_dist": float(np.mean(d)),
        "median_dist": float(np.median(d)),
        "p95_dist": float(np.quantile(d, .95)),
        "max_dist": float(np.max(d)),
        "exact_rate": float((d <= EXACT_DISTANCE).mean()),
    }


def prepare_binary_target(s):
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().all() and set(x.unique()).issubset({0, 1}):
        return x.astype(int)
    return None


def assignment_cost(
    Z_source,
    T_target,
    source_y=None,
    target_y=None,
    label_mode="none",
):
    """
    rectangular Hungarian:
    target 1211 rows -> source candidate rows

    label_mode:
      none      : 센서만
      same      : source target == target target 선호
      inverted  : source target == 1-target 선호
    """
    A = T_target.to_numpy(float)
    B = Z_source.to_numpy(float)

    C = cdist(A, B, metric="euclidean")

    if (
        label_mode != "none"
        and source_y is not None
        and target_y is not None
    ):
        sy = np.asarray(source_y)
        ty = np.asarray(target_y)

        if label_mode == "same":
            expected = ty
        elif label_mode == "inverted":
            expected = 1 - ty
        else:
            expected = None

        if expected is not None:
            mismatch = expected[:, None] != sy[None, :]
            C = C + mismatch.astype(float) * LABEL_MISMATCH_PENALTY

    return C


def hungarian_map(
    source_df,
    target_df,
    feature_cols,
    target_feature_cols,
    label_mode="none",
):
    Z, mu, sd = scale_subset(source_df, feature_cols)

    Z2 = Z.copy()
    Z2.columns = target_feature_cols

    T = target_df[target_feature_cols].astype(float)

    ys_col = target_col(source_df)
    yt_col = target_col(target_df)

    sy = prepare_binary_target(source_df[ys_col]) if ys_col else None
    ty = prepare_binary_target(target_df[yt_col]) if yt_col else None

    C = assignment_cost(
        Z2,
        T,
        source_y=sy.to_numpy() if sy is not None else None,
        target_y=ty.to_numpy() if ty is not None else None,
        label_mode=label_mode,
    )

    row_ind, col_ind = linear_sum_assignment(C)

    # row_ind = target row position
    # col_ind = source row position
    raw_dist = cdist(
        T.to_numpy(float),
        Z2.to_numpy(float),
        metric="euclidean",
    )[row_ind, col_ind]

    mapping = pd.DataFrame({
        "target_pos": row_ind,
        "source_pos": col_ind,
        "target_row_index": target_df.index.to_numpy()[row_ind],
        "source_row_index": source_df.index.to_numpy()[col_ind],
        "sensor_distance": raw_dist,
    })

    if sy is not None and ty is not None:
        mapped_sy = sy.to_numpy()[col_ind]
        mapped_ty = ty.to_numpy()[row_ind]

        mapping["source_target"] = mapped_sy
        mapping["target_target"] = mapped_ty
        mapping["target_same"] = mapped_sy == mapped_ty
        mapping["target_inverted"] = mapped_sy == (1 - mapped_ty)

    summary = {
        "label_mode": label_mode,
        "n": len(mapping),
        "median_dist": float(mapping["sensor_distance"].median()),
        "mean_dist": float(mapping["sensor_distance"].mean()),
        "p95_dist": float(mapping["sensor_distance"].quantile(.95)),
        "max_dist": float(mapping["sensor_distance"].max()),
        "exact_rate": float((mapping["sensor_distance"] <= EXACT_DISTANCE).mean()),
    }

    if "target_same" in mapping.columns:
        summary["target_same_rate"] = float(mapping["target_same"].mean())
        summary["target_inverted_rate"] = float(mapping["target_inverted"].mean())

    return mapping, summary, mu, sd


def score_exact_subset(
    subset,
    target,
    source_feature_cols,
    target_feature_cols,
    subset_name,
):
    """
    정확히 1211행짜리 후보 subset 평가.
    direct order + Hungarian(same/inverted/none) 모두 확인.
    """
    records = []
    outputs = {}

    Z, mu, sd = scale_subset(subset, source_feature_cols)
    Z_cmp = Z.copy()
    Z_cmp.columns = target_feature_cols

    direct = direct_error(
        Z_cmp.reset_index(drop=True),
        target[target_feature_cols].astype(float).reset_index(drop=True),
    )

    if direct is not None:
        records.append({
            "subset": subset_name,
            "mapping": "direct_order",
            **direct,
        })

    for mode in ["none", "same", "inverted"]:
        mapping, summary, mu2, sd2 = hungarian_map(
            subset,
            target,
            source_feature_cols,
            target_feature_cols,
            label_mode=mode,
        )

        rec = {
            "subset": subset_name,
            "mapping": f"hungarian_{mode}",
            "mean_dist": summary["mean_dist"],
            "median_dist": summary["median_dist"],
            "p95_dist": summary["p95_dist"],
            "max_dist": summary["max_dist"],
            "exact_rate": summary["exact_rate"],
            "target_same_rate": summary.get("target_same_rate", np.nan),
            "target_inverted_rate": summary.get("target_inverted_rate", np.nan),
        }
        records.append(rec)
        outputs[f"hungarian_{mode}"] = (mapping, mu2, sd2)

    return pd.DataFrame(records), outputs


def iterative_assignment(
    source_all,
    target,
    source_feature_cols,
    target_feature_cols,
    label_mode="none",
    max_iter=20,
):
    """
    exact filter가 안 보일 때 사용하는 fallback.

    1) 전체 1425 기준 scaler
    2) target 1211 -> source 1425 Hungarian assignment
    3) 선택된 1211만으로 scaler 재계산
    4) 반복
    """
    selected_idx = source_all.index.to_numpy()
    current_subset = source_all.copy()

    history = []
    prev_selected = None
    final_mapping = None
    final_mu = None
    final_sd = None

    for it in range(1, max_iter + 1):
        # 현재 선택 subset으로 scaler 파라미터 추정
        _, mu, sd = scale_subset(
            current_subset,
            source_feature_cols,
        )

        # 그 scaler로 "전체 1425"를 변환
        X_all = source_all[source_feature_cols].astype(float)
        Z_all = pd.DataFrame(
            index=source_all.index,
            columns=source_feature_cols,
            dtype=float,
        )

        for c in source_feature_cols:
            if pd.isna(sd[c]) or abs(sd[c]) < 1e-12:
                Z_all[c] = 0.0
            else:
                Z_all[c] = (X_all[c] - mu[c]) / sd[c]

        Z_all.columns = target_feature_cols
        T = target[target_feature_cols].astype(float)

        ys_col = target_col(source_all)
        yt_col = target_col(target)

        sy = prepare_binary_target(source_all[ys_col]) if ys_col else None
        ty = prepare_binary_target(target[yt_col]) if yt_col else None

        C = assignment_cost(
            Z_all,
            T,
            source_y=sy.to_numpy() if sy is not None else None,
            target_y=ty.to_numpy() if ty is not None else None,
            label_mode=label_mode,
        )

        row_ind, col_ind = linear_sum_assignment(C)

        raw_C = cdist(
            T.to_numpy(float),
            Z_all.to_numpy(float),
            metric="euclidean",
        )
        d = raw_C[row_ind, col_ind]

        selected_positions = np.sort(col_ind)
        selected_source_idx = source_all.index.to_numpy()[selected_positions]

        hist = {
            "iter": it,
            "label_mode": label_mode,
            "median_dist": float(np.median(d)),
            "mean_dist": float(np.mean(d)),
            "p95_dist": float(np.quantile(d, .95)),
            "max_dist": float(np.max(d)),
            "exact_rate": float((d <= EXACT_DISTANCE).mean()),
            "selected_changed": (
                np.nan if prev_selected is None
                else int(
                    len(
                        set(selected_source_idx)
                        .symmetric_difference(set(prev_selected))
                    )
                )
            ),
        }
        history.append(hist)

        mapping = pd.DataFrame({
            "target_pos": row_ind,
            "source_pos": col_ind,
            "target_row_index": target.index.to_numpy()[row_ind],
            "source_row_index": source_all.index.to_numpy()[col_ind],
            "sensor_distance": d,
        })

        if sy is not None and ty is not None:
            mapped_sy = sy.to_numpy()[col_ind]
            mapped_ty = ty.to_numpy()[row_ind]
            mapping["source_target"] = mapped_sy
            mapping["target_target"] = mapped_ty
            mapping["target_same"] = mapped_sy == mapped_ty
            mapping["target_inverted"] = mapped_sy == (1 - mapped_ty)

        final_mapping = mapping
        final_mu = mu
        final_sd = sd

        if prev_selected is not None and set(selected_source_idx) == set(prev_selected):
            break

        prev_selected = selected_source_idx
        current_subset = source_all.loc[selected_source_idx].copy()

    return (
        pd.DataFrame(history),
        final_mapping,
        current_subset,
        final_mu,
        final_sd,
    )


def affine_column_check(
    subset,
    target,
    mapping,
    source_feature_cols,
    target_feature_cols,
):
    """
    매핑된 raw x와 target z에 대해 x = intercept + slope*z를 fitting.
    StandardScaler가 맞으면:
      intercept ~= subset mean
      slope ~= subset std(ddof=0)
      residual ~= 0
    """
    rows = []

    map2 = mapping.sort_values("target_pos")

    for cs, ct in zip(source_feature_cols, target_feature_cols):
        x = subset.loc[map2["source_row_index"], cs].to_numpy(float)
        z = target.loc[map2["target_row_index"], ct].to_numpy(float)

        if np.std(z) < 1e-12:
            slope = np.nan
            intercept = float(np.mean(x))
            pred = np.repeat(intercept, len(x))
        else:
            slope, intercept = np.polyfit(z, x, deg=1)
            pred = intercept + slope * z

        resid = x - pred

        subset_mean = subset[cs].astype(float).mean()
        subset_std0 = subset[cs].astype(float).std(ddof=0)

        rows.append({
            "feature": cs,
            "affine_intercept": intercept,
            "subset_mean": subset_mean,
            "intercept_minus_mean": intercept - subset_mean,
            "affine_slope": slope,
            "subset_std0": subset_std0,
            "slope_minus_std": (
                slope - subset_std0 if pd.notna(slope) else np.nan
            ),
            "resid_mae": float(np.mean(np.abs(resid))),
            "resid_max_abs": float(np.max(np.abs(resid))),
        })

    return pd.DataFrame(rows)


# =============================================================================
# 1. 로드
# =============================================================================

banner("CN7 02 - 1425 -> 1211 원본행 복원")

source_path = find_file(BASE_DIR, SOURCE_NAMES)
target_path = find_file(BASE_DIR, TARGET_NAMES)

if source_path is None:
    raise FileNotFoundError(
        f"moldset_labeled.csv를 찾지 못했습니다.\nBASE_DIR={BASE_DIR}"
    )

if target_path is None:
    raise FileNotFoundError(
        f"moldset_labeled_cn7.csv를 찾지 못했습니다.\nBASE_DIR={BASE_DIR}"
    )

source_full = read_csv_flexible(source_path)
target = read_csv_flexible(target_path)

print("source:", source_path)
print("target:", target_path)
print("source full:", source_full.shape)
print("target     :", target.shape)

mask_cn7 = get_cn7_mask(source_full)
source = source_full.loc[mask_cn7].copy()

print("source CN7 :", source.shape)

if len(target) >= len(source):
    raise RuntimeError(
        f"예상과 다른 행 수입니다. source CN7={len(source)}, target={len(target)}"
    )

N_TARGET = len(target)

# 원본 index 고정
source["__source_row_index__"] = source.index

# Side 생성
part_name_col = find_col(source, ["PART_NAME"])
if part_name_col is not None:
    source["__Side__"] = extract_side(source[part_name_col])
else:
    source["__Side__"] = "UNKNOWN"

# =============================================================================
# 2. 공통 센서
# =============================================================================

banner("1) 공통 공정 변수")

pairs = common_numeric_features(source, target)

source_features = [x[0] for x in pairs]
target_features = [x[1] for x in pairs]

print("공통 numeric feature:", len(pairs))
for a, b in pairs:
    print(f"  {a} <-> {b}")

if len(pairs) < 5:
    raise RuntimeError("공통 numeric feature가 너무 적습니다.")

# =============================================================================
# 3. target에서 상수인 컬럼 -> 1425->1211 필터 단서
# =============================================================================

banner("2) Target 상수 컬럼으로 필터 규칙 탐색")

constant_features = []
constant_rule_candidates = []

for cs, ct in pairs:
    t_unique = target[ct].nunique(dropna=False)
    t_std = pd.to_numeric(target[ct], errors="coerce").std(ddof=0)

    if t_unique <= 1 or (pd.notna(t_std) and abs(t_std) < 1e-12):
        constant_features.append((cs, ct))

        vc = source[cs].value_counts(dropna=False)

        print(f"\nTarget constant: {ct}")
        print(f"Source {cs} value counts:")
        print(vc.head(20).to_string())

        for val, count in vc.items():
            if int(count) == N_TARGET:
                idx = source.index[source[cs].eq(val)].to_numpy()
                constant_rule_candidates.append({
                    "rule_name": f"{cs} == {val}",
                    "index": idx,
                    "source_feature": cs,
                    "value": val,
                })
                print(
                    f">>> EXACT {N_TARGET}개 규칙 발견: "
                    f"{cs} == {val}"
                )

print("\nTarget constant features:", [x[0] for x in constant_features])

# 여러 상수 컬럼의 값 조합도 탐색
if len(constant_features) >= 2:
    ccols = [x[0] for x in constant_features]

    combo_counts = (
        source.groupby(ccols, dropna=False)
        .size()
        .sort_values(ascending=False)
    )

    combo_df = combo_counts.reset_index(name="count")
    combo_df.to_csv(
        OUT_DIR / "constant_feature_combinations.csv",
        index=False,
        encoding="utf-8-sig",
    )

    for _, r in combo_df.iterrows():
        if int(r["count"]) == N_TARGET:
            m = pd.Series(True, index=source.index)
            desc = []

            for c in ccols:
                v = r[c]

                if pd.isna(v):
                    m &= source[c].isna()
                    desc.append(f"{c}=NaN")
                else:
                    m &= source[c].eq(v)
                    desc.append(f"{c}={v}")

            constant_rule_candidates.append({
                "rule_name": " & ".join(desc),
                "index": source.index[m].to_numpy(),
                "source_feature": "|".join(ccols),
                "value": "combination",
            })

# =============================================================================
# 4. metadata 단일 조건으로 1211개가 되는 규칙 탐색
# =============================================================================

banner("3) Metadata 단일/조합 필터 규칙 탐색")

metadata_cols = []

for c in source.columns:
    if c in source_features:
        continue
    if c in ["__source_row_index__"]:
        continue

    # 너무 고유한 자유텍스트/ID는 단일값 필터에서 제외하지 않되 출력만 제한
    metadata_cols.append(c)

metadata_exact_candidates = []

for c in metadata_cols:
    try:
        vc = source[c].value_counts(dropna=False)

        exact_vals = vc[vc == N_TARGET]

        if len(exact_vals):
            print(f"\n[{c}] 정확히 {N_TARGET}행 만드는 값:")
            print(exact_vals.to_string())

            for val, count in exact_vals.items():
                if pd.isna(val):
                    idx = source.index[source[c].isna()].to_numpy()
                    rule = f"{c} is NaN"
                else:
                    idx = source.index[source[c].eq(val)].to_numpy()
                    rule = f"{c} == {val}"

                metadata_exact_candidates.append({
                    "rule_name": rule,
                    "index": idx,
                    "source_feature": c,
                    "value": val,
                })
    except Exception:
        pass

# 날짜/시리얼/파트별 카운트 저장
meta_count_tables = []

for c in metadata_cols:
    nuniq = source[c].nunique(dropna=False)

    if nuniq <= 100:
        tmp = (
            source[c]
            .value_counts(dropna=False)
            .rename_axis("value")
            .reset_index(name="count")
        )
        tmp.insert(0, "column", c)
        meta_count_tables.append(tmp)

if meta_count_tables:
    pd.concat(meta_count_tables, ignore_index=True).to_csv(
        OUT_DIR / "metadata_value_counts.csv",
        index=False,
        encoding="utf-8-sig",
    )

# 후보 합치기 / 중복 rule 제거
all_rule_candidates = constant_rule_candidates + metadata_exact_candidates

dedup = {}
for r in all_rule_candidates:
    key = tuple(sorted(map(int, r["index"])))
    if len(key) == N_TARGET:
        dedup.setdefault(key, r)

all_rule_candidates = list(dedup.values())

print(f"\n정확히 {N_TARGET}행을 만드는 단순 규칙 후보: {len(all_rule_candidates)}개")

for i, r in enumerate(all_rule_candidates, 1):
    print(f"{i:2d}. {r['rule_name']}")

# =============================================================================
# 5. exact 1211 subset 후보들을 실제 z-score 재현으로 검증
# =============================================================================

banner("4) 후보 subset의 StandardScaler 재현 검증")

candidate_score_frames = []
candidate_outputs = {}

for i, r in enumerate(all_rule_candidates, 1):
    subset = source.loc[r["index"]].copy()

    if len(subset) != N_TARGET:
        continue

    name = f"rule_{i:02d}__{r['rule_name']}"

    scores, outputs = score_exact_subset(
        subset,
        target,
        source_features,
        target_features,
        subset_name=name,
    )

    candidate_score_frames.append(scores)
    candidate_outputs[name] = {
        "subset": subset,
        "outputs": outputs,
        "rule": r,
    }

    print(f"\n[{name}]")
    print(
        scores[
            [
                "mapping",
                "median_dist",
                "p95_dist",
                "max_dist",
                "exact_rate",
                "target_same_rate",
                "target_inverted_rate",
            ]
        ]
        .round(8)
        .to_string(index=False)
    )

if candidate_score_frames:
    candidate_scores = pd.concat(candidate_score_frames, ignore_index=True)

    candidate_scores = candidate_scores.sort_values(
        ["median_dist", "p95_dist", "max_dist"]
    )

    candidate_scores.to_csv(
        OUT_DIR / "exact_subset_candidate_scores.csv",
        index=False,
        encoding="utf-8-sig",
    )
else:
    candidate_scores = pd.DataFrame()
    print("정확히 1211행을 만드는 단순 규칙 후보가 없습니다.")

# =============================================================================
# 6. simple exact rule이 약하면 iterative Hungarian fallback
# =============================================================================

banner("5) Iterative assignment fallback")

iter_results = []
iter_outputs = {}

# exact candidate가 아주 좋으면 fallback도 참고용으로는 돌림.
# source 1425 x target 1211 정도라 계산 가능.
for mode in ["none", "same", "inverted"]:
    history, mapping, subset, mu, sd = iterative_assignment(
        source_all=source,
        target=target,
        source_feature_cols=source_features,
        target_feature_cols=target_features,
        label_mode=mode,
        max_iter=MAX_ITER,
    )

    history.to_csv(
        OUT_DIR / f"iterative_history__{mode}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    final = history.iloc[-1].to_dict()
    final["method"] = f"iterative_{mode}"
    iter_results.append(final)

    iter_outputs[mode] = {
        "history": history,
        "mapping": mapping,
        "subset": subset,
        "mu": mu,
        "sd": sd,
    }

    print(f"\n[{mode}]")
    print(history.round(8).to_string(index=False))

iter_df = pd.DataFrame(iter_results)
iter_df.to_csv(
    OUT_DIR / "iterative_assignment_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 7. 최종 후보 선택
# =============================================================================

banner("6) 최종 복원 후보 선택")

final_kind = None
final_name = None
final_subset = None
final_mapping = None
final_mu = None
final_sd = None
final_metrics = None

# 1순위: exact rule + Hungarian 중 최저 거리
if len(candidate_scores):
    h = candidate_scores[
        candidate_scores["mapping"].str.startswith("hungarian")
    ].copy()

    if len(h):
        best = h.sort_values(
            ["median_dist", "p95_dist", "max_dist"]
        ).iloc[0]

        final_kind = "exact_rule"
        final_name = best["subset"]
        final_metrics = best.to_dict()

        obj = candidate_outputs[final_name]
        final_subset = obj["subset"]

        mapping_key = best["mapping"]
        final_mapping, final_mu, final_sd = obj["outputs"][mapping_key]

# 2순위: iterative
if final_mapping is None:
    best_iter = iter_df.sort_values(
        ["median_dist", "p95_dist", "max_dist"]
    ).iloc[0]

    mode = str(best_iter["label_mode"])

    final_kind = "iterative"
    final_name = f"iterative_{mode}"
    final_metrics = best_iter.to_dict()

    obj = iter_outputs[mode]
    final_subset = obj["subset"]
    final_mapping = obj["mapping"]
    final_mu = obj["mu"]
    final_sd = obj["sd"]

print("선택 method:", final_kind)
print("선택 name  :", final_name)
print("metrics    :", final_metrics)

# =============================================================================
# 8. 최종 매핑을 target 행 순서로 정렬 + 원본 metadata 붙이기
# =============================================================================

banner("7) 원본 metadata 복원")

final_mapping = final_mapping.sort_values("target_pos").reset_index(drop=True)

# target 원본 순서
target_ordered = target.loc[
    final_mapping["target_row_index"]
].copy().reset_index(drop=False)
target_ordered = target_ordered.rename(
    columns={"index": "__target_original_index__"}
)

source_ordered = source.loc[
    final_mapping["source_row_index"]
].copy().reset_index(drop=False)
source_ordered = source_ordered.rename(
    columns={"index": "__source_original_index__"}
)

# 중복 이름 충돌 방지를 위해 metadata 위주만 가져오기
metadata_restore_cols = []

for c in source_ordered.columns:
    if c in source_features:
        continue
    if c == target_col(source_ordered):
        continue
    metadata_restore_cols.append(c)

restored = target_ordered.copy()

for c in metadata_restore_cols:
    out_c = c
    if out_c in restored.columns:
        out_c = f"RAW__{c}"
    restored[out_c] = source_ordered[c].to_numpy()

# source target도 별도 보존
src_target_col = target_col(source_ordered)
tgt_target_col = target_col(target_ordered)

if src_target_col is not None:
    restored["__source_PassOrFail__"] = source_ordered[src_target_col].to_numpy()

if "__Side__" in source_ordered.columns:
    restored["Side"] = source_ordered["__Side__"].to_numpy()

# mapping distance
restored["__sensor_match_distance__"] = final_mapping[
    "sensor_distance"
].to_numpy()

restored.to_csv(
    OUT_DIR / "cn7_1211_restored.csv",
    index=False,
    encoding="utf-8-sig",
)

final_mapping.to_csv(
    OUT_DIR / "cn7_1211_row_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

print("복원 데이터 shape:", restored.shape)
print("sensor distance:")
print(restored["__sensor_match_distance__"].describe().to_string())

# =============================================================================
# 9. scaler 정확성 검증
# =============================================================================

banner("8) StandardScaler 정확성 검증")

# mapping에 포함된 source row 전체가 final subset이어야 함
selected_source_indices = final_mapping["source_row_index"].to_numpy()
selected_source = source.loc[selected_source_indices].copy()

# 중복 없이 target size인지 확인
print("선택 source 행 수:", len(selected_source_indices))
print("선택 source unique index:", len(np.unique(selected_source_indices)))

# subset 전체에서 scaler 계산
scaler_subset = source.loc[
    np.unique(selected_source_indices)
].copy()

affine = affine_column_check(
    subset=scaler_subset,
    target=target,
    mapping=final_mapping,
    source_feature_cols=source_features,
    target_feature_cols=target_features,
)

affine.to_csv(
    OUT_DIR / "scaler_reconstruction_check.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    affine[
        [
            "feature",
            "intercept_minus_mean",
            "slope_minus_std",
            "resid_mae",
            "resid_max_abs",
        ]
    ]
    .round(10)
    .to_string(index=False)
)

# scaler 파라미터 저장
_, actual_mu, actual_sd = scale_subset(
    scaler_subset,
    source_features,
)

scaler_params = pd.DataFrame({
    "feature": source_features,
    "mean_raw": [actual_mu[c] for c in source_features],
    "std0_raw": [actual_sd[c] for c in source_features],
})

scaler_params.to_csv(
    OUT_DIR / "recovered_scaler_parameters.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 10. Side / target / 시간 진단
# =============================================================================

banner("9) 복원 metadata 진단")

# PART_NAME / Side
if "Side" in restored.columns:
    print("\nSide counts:")
    print(restored["Side"].value_counts(dropna=False).to_string())

    if tgt_target_col is not None:
        print("\nTarget x Side:")
        side_ct = pd.crosstab(
            restored["Side"],
            restored[tgt_target_col],
            margins=True,
        )
        print(side_ct.to_string())
        side_ct.to_csv(
            OUT_DIR / "target_by_side.csv",
            encoding="utf-8-sig",
        )

# source target vs target target
if tgt_target_col is not None and "__source_PassOrFail__" in restored.columns:
    a = pd.to_numeric(
        restored["__source_PassOrFail__"],
        errors="coerce",
    )
    b = pd.to_numeric(
        restored[tgt_target_col],
        errors="coerce",
    )

    valid = a.notna() & b.notna()

    if valid.any():
        same = (a[valid].to_numpy() == b[valid].to_numpy()).mean()
        inv = ((1 - a[valid].to_numpy()) == b[valid].to_numpy()).mean()

        print("\nSource/target PassOrFail:")
        print("same rate    :", same)
        print("inverted rate:", inv)

# 시간
time_col = find_col(restored, ["TimeStamp", "RAW__TimeStamp"])

if time_col is not None:
    dt = pd.to_datetime(restored[time_col], errors="coerce")

    print("\nTimeStamp:")
    print("min:", dt.min())
    print("max:", dt.max())
    print("parsed:", dt.notna().sum(), "/", len(dt))

# serial
serial_col = find_col(
    restored,
    ["PART_FACT_SERIAL", "RAW__PART_FACT_SERIAL"],
)

if serial_col is not None:
    print("\nPART_FACT_SERIAL:")
    print(restored[serial_col].value_counts(dropna=False).to_string())

# plan date
plan_col = find_col(
    restored,
    ["PART_FACT_PLAN_DATE", "RAW__PART_FACT_PLAN_DATE"],
)

if plan_col is not None:
    print("\nPART_FACT_PLAN_DATE:")
    print(restored[plan_col].value_counts(dropna=False).to_string())

# =============================================================================
# 11. 제외된 214행과 선택된 1211행 비교
# =============================================================================

banner("10) 선택 1211 vs 제외 214 비교")

selected_idx_set = set(
    map(int, final_mapping["source_row_index"].tolist())
)

source_compare = source.copy()
source_compare["__selected_1211__"] = source_compare.index.map(
    lambda x: int(x) in selected_idx_set
)

print(source_compare["__selected_1211__"].value_counts().to_string())

# metadata별 선택률
filter_diag_rows = []

for c in source.columns:
    if c in source_features:
        continue
    if c.startswith("__"):
        continue

    try:
        g = (
            source_compare
            .groupby(c, dropna=False)["__selected_1211__"]
            .agg(["count", "sum", "mean"])
            .reset_index()
        )
        g["column"] = c
        g = g.rename(
            columns={
                "sum": "selected_count",
                "mean": "selected_rate",
            }
        )

        if len(g) <= 200:
            filter_diag_rows.append(
                g[
                    [
                        "column",
                        c,
                        "count",
                        "selected_count",
                        "selected_rate",
                    ]
                ].rename(columns={c: "value"})
            )
    except Exception:
        pass

if filter_diag_rows:
    filter_diag = pd.concat(
        filter_diag_rows,
        ignore_index=True,
    )
    filter_diag.to_csv(
        OUT_DIR / "selected_vs_excluded_metadata.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 0% 또는 100% 선택되는 범주부터 출력
    strong = filter_diag[
        (filter_diag["selected_rate"] <= 0.001)
        | (filter_diag["selected_rate"] >= 0.999)
    ].copy()

    print("\n선택/제외를 완전히 가르는 metadata 값:")
    if len(strong):
        print(
            strong.sort_values(
                ["column", "selected_rate", "count"]
            )
            .to_string(index=False)
        )
    else:
        print("없음")

# 센서별 selected/excluded raw 분포
sensor_diag = []

for c in source_features:
    a = source_compare.loc[
        source_compare["__selected_1211__"], c
    ].astype(float)

    b = source_compare.loc[
        ~source_compare["__selected_1211__"], c
    ].astype(float)

    sensor_diag.append({
        "feature": c,
        "selected_n": len(a),
        "excluded_n": len(b),
        "selected_unique": a.nunique(dropna=False),
        "excluded_unique": b.nunique(dropna=False),
        "selected_mean": a.mean(),
        "excluded_mean": b.mean(),
        "selected_min": a.min(),
        "selected_max": a.max(),
        "excluded_min": b.min(),
        "excluded_max": b.max(),
    })

pd.DataFrame(sensor_diag).to_csv(
    OUT_DIR / "selected_vs_excluded_sensors.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 12. 자동 요약
# =============================================================================

banner("11) 자동 복원 요약")

summary = []

summary.append(
    f"- source CN7: {len(source)}행 -> target CN7: {len(target)}행"
)
summary.append(
    f"- 최종 복원 방법: {final_kind} / {final_name}"
)
summary.append(
    f"- sensor distance median: "
    f"{restored['__sensor_match_distance__'].median():.10g}"
)
summary.append(
    f"- sensor distance p95: "
    f"{restored['__sensor_match_distance__'].quantile(.95):.10g}"
)
summary.append(
    f"- exact(<= {EXACT_DISTANCE}) rate: "
    f"{(restored['__sensor_match_distance__'] <= EXACT_DISTANCE).mean():.3%}"
)

if "Side" in restored.columns:
    vc = restored["Side"].value_counts().to_dict()
    summary.append(
        f"- Side counts: {vc}"
    )

if tgt_target_col is not None and "__source_PassOrFail__" in restored.columns:
    a = pd.to_numeric(
        restored["__source_PassOrFail__"],
        errors="coerce",
    )
    b = pd.to_numeric(
        restored[tgt_target_col],
        errors="coerce",
    )
    valid = a.notna() & b.notna()

    if valid.any():
        same = (
            a[valid].to_numpy() == b[valid].to_numpy()
        ).mean()
        inv = (
            (1 - a[valid].to_numpy()) == b[valid].to_numpy()
        ).mean()

        summary.append(
            f"- PassOrFail same rate: {same:.6f}"
        )
        summary.append(
            f"- PassOrFail inverted rate: {inv:.6f}"
        )

if time_col is not None:
    dt = pd.to_datetime(
        restored[time_col],
        errors="coerce",
    )
    summary.append(
        f"- Time range: {dt.min()} ~ {dt.max()}"
    )

for x in summary:
    print(x)

with open(
    OUT_DIR / "restore_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(summary))

# =============================================================================
# 13. 완료
# =============================================================================

banner("완료")

print("결과 폴더:", OUT_DIR.resolve())
print()
print("핵심 결과 파일:")
print("  1) exact_subset_candidate_scores.csv")
print("  2) iterative_assignment_summary.csv")
print("  3) cn7_1211_row_mapping.csv")
print("  4) cn7_1211_restored.csv")
print("  5) scaler_reconstruction_check.csv")
print("  6) recovered_scaler_parameters.csv")
print("  7) selected_vs_excluded_metadata.csv")
print("  8) selected_vs_excluded_sensors.csv")
print("  9) target_by_side.csv")
print(" 10) restore_summary.txt")
print()
print(
    "다음 단계에서는 restore_summary.txt와 "
    "cn7_1211_restored.csv를 기준으로 EDA를 시작합니다."
)
